# AlignVid object-addition pilot: generation on Kaggle

Runs `gen/run_pilot.py` from the pilot package (uploaded as a private Kaggle dataset).
Settings for this notebook: **Accelerator = GPU** (prefer any Ampere-or-newer GPU if offered, else T4 x2), **Internet = On**, Persistence = off.
Session 1: run the cells one by one up to the dry run. Session 2: *Save Version → Save & Run All* for the full generation (up to 12 h, headless).

Heavy files (code, weights ≈ 35 GB) live in `/kaggle/tmp/pilot`, which is not saved. Finished videos, sidecars and logs are copied to `/kaggle/working` as they appear, which is saved as the notebook output.

In [ ]:
# 1. unpack the package; restore results from a previous version of this notebook, if attached as input
import glob, os, shutil, subprocess, zipfile
PKG = sorted(glob.glob('/kaggle/input/*/pilot_package.zip'))[0]          # the uploaded dataset
ROOT = '/kaggle/tmp/pilot/pilot_package'
if not os.path.exists(ROOT):
    os.makedirs('/kaggle/tmp/pilot', exist_ok=True)
    zipfile.ZipFile(PKG).extractall('/kaggle/tmp/pilot')
for prev in glob.glob('/kaggle/input/*/outputs') + glob.glob('/kaggle/input/*/logs'):   # previous notebook output
    shutil.copytree(prev, os.path.join(ROOT, os.path.basename(prev)), dirs_exist_ok=True)
    print('restored', prev)
os.chdir(ROOT); print(open('VERSION.json').read())
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout.splitlines()[:10])
print(subprocess.run(['df', '-h', '/kaggle/tmp', '/kaggle/working'], capture_output=True, text=True).stdout)
print(subprocess.run(['free', '-g'], capture_output=True, text=True).stdout)

In [ ]:
# 2. packages: keep Kaggle's torch, install FramePack's pins (log in /kaggle/working/logs/pip.txt)
import os, subprocess, sys
os.makedirs('/kaggle/working/logs', exist_ok=True)
with open('/kaggle/working/logs/pip.txt', 'w') as f:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'gen/requirements-laptop.txt'], stdout=f, stderr=subprocess.STDOUT)
print('pip exit', r.returncode)
print(subprocess.run([sys.executable, '-c', 'import torch, transformers, diffusers, numpy; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0), transformers.__version__, diffusers.__version__, numpy.__version__)'], capture_output=True, text=True))

In [ ]:
# 3. C1 environment check (GPU, bf16 matmul, versions) -> logs/env_kaggle.*
!PILOT_HOST=kaggle python gen/run_pilot.py --stage env --sync-dir /kaggle/working

In [ ]:
# 4. patch (gloo fallback; no-op on Linux with NCCL) and weight download (~35 GB, resumable)
!PILOT_HOST=kaggle python gen/run_pilot.py --stage patch --sync-dir /kaggle/working
!PILOT_HOST=kaggle python gen/run_pilot.py --stage weights --sync-dir /kaggle/working
!du -sh external/AlignVid/models/FramePack/hf_download; df -h /kaggle/tmp | tail -1

In [ ]:
# 5. C3 dry run: vanilla sample_240, 2 s (then 1 s) at --gpu_mem 6.  Watch logs/run_pilot.log; FramePack's own output is in logs/vanilla_sample_240__len*.json
!PILOT_HOST=kaggle python gen/run_pilot.py --stage dryrun --quiet --sync-dir /kaggle/working
!tail -5 logs/run_pilot.log; cat logs/run_state.json

If cell 5 printed `done in ... s`, the pipeline works here. For the full run use **Save Version → Save & Run All**: cells 1–5 are skipped quickly (results restored, weights re-downloaded), then cell 6 runs the length probe and all arms. Attach the previous version's output as an input (*Add Input → Your Work → this notebook*) so finished videos are not regenerated.

In [ ]:
# 6. C4: length probe (6.0 → 3.0 → 2.0) and all arms (vanilla, alignvid, noedit), one process per video; results zip in /kaggle/working
!PILOT_HOST=kaggle python gen/run_pilot.py --stage length --quiet --sync-dir /kaggle/working
!PILOT_HOST=kaggle python gen/run_pilot.py --stage arms --quiet --sync-dir /kaggle/working
!PILOT_HOST=kaggle python gen/run_pilot.py --stage pack --sync-dir /kaggle/working
!grep -E "chosen|done in|FAILED|arms:|packed" logs/run_pilot.log; ls -la /kaggle/working /kaggle/working/outputs/* 2>/dev/null | head -40